# Project 1: Messy Data Cleanup (Titanic passengers)
Goal: take a raw dataset with missing values, inconsistent text and duplicates, and produce a clean, documented dataset with a before/after report.

Data: public Titanic passenger data (seaborn-data on GitHub).

In [1]:
import pandas as pd, numpy as np
raw = pd.read_csv("titanic.csv")
df = raw.copy()
print(raw.shape)
raw.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 1. Audit (before)

In [2]:
audit = pd.DataFrame({"dtype":raw.dtypes.astype(str),"missing":raw.isna().sum(),"missing_%":(raw.isna().mean()*100).round(1),"unique":raw.nunique()})
print("duplicate rows:", raw.duplicated().sum())
audit

duplicate rows: 107


,dtype,missing,missing_%,unique
survived,int64,0,0.0,2
pclass,int64,0,0.0,3
sex,object,0,0.0,2
age,float64,177,19.9,88
sibsp,int64,0,0.0,7
parch,int64,0,0.0,7
fare,float64,0,0.0,248
embarked,object,2,0.2,3
class,object,0,0.0,3
who,object,0,0.0,3


## 2. Cleaning steps
- Check duplicates: rows are kept because there is no ID column (identical rows can be different passengers)
- Standardize text (strip, lowercase, consistent labels)
- `deck`: 77% missing, so mark as "unknown" instead of dropping rows
- `age`: fill with median by passenger class and sex, add an `age_was_missing` flag
- `embarked` / `embark_town`: fill the 2 missing with the mode
- Fix types (category, bool)
- Drop `alive` and `adult_male` (duplicates of other columns)

In [3]:
# NOTE: 107 rows look like duplicates, but there is no passenger ID, so identical rows can be different people. Kept on purpose.
for c in df.select_dtypes("object"):
    df[c] = df[c].str.strip().str.lower()
df["age_was_missing"] = df["age"].isna()
df["age"] = df.groupby(["pclass","sex"])["age"].transform(lambda s: s.fillna(s.median()))
df["deck"] = df["deck"].astype("object").fillna("unknown")
for c in ["embarked","embark_town"]:
    df[c] = df[c].fillna(df[c].mode()[0])
df = df.drop(columns=["alive","adult_male"])
for c in ["sex","embarked","class","who","deck","embark_town"]:
    df[c] = df[c].astype("category")
df["alone"] = df["alone"].astype(bool)
df.isna().sum().sum()

np.int64(0)

## 3. Validation (after)

In [4]:
assert df.isna().sum().sum()==0
assert df["age"].between(0,100).all()
assert df["fare"].ge(0).all()
rep = pd.DataFrame({"before":[raw.shape[0],raw.isna().sum().sum(),raw.duplicated().sum()],"after":[df.shape[0],df.isna().sum().sum(),df.duplicated().sum()]},index=["rows","missing cells","duplicate rows"])
rep

,before,after
rows,891,891
missing cells,869,0
duplicate rows,107,107


In [5]:
df.to_csv("titanic_clean.csv",index=False)
print("saved", df.shape)

saved (891, 14)


## Summary
Result: zero missing cells, documented decisions, a flag column that keeps the information about what was imputed, and a CSV ready for modelling.